<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a> 一书的补充代码，作者 <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 附录 E：批处理与面向吞吐量的执行

本笔记本使用的软件包：

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",  # for download functions
    "torch",
    "tokenizers"
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.17
torch version: 2.10.0
tokenizers version: 0.21.4


- 在主要章节中，我们通常一次处理一个样本
- 这使得代码紧凑且更容易理解
- 但同时，代码已经非常昂贵，因此添加批处理支持由于硬件和资源限制会带来很少的收益
- 然而，在某些情况下，能够以批处理模式运行代码仍然是有用的
- 本附录解释了批处理执行的广泛思想，并展示了如何使用补充材料中的代码在不同章节中使用它

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-e/Appendix_E_F01_raschka.webp" width="400px">

&nbsp;
## E.1 为什么批处理有帮助

- 有两个不同的性能目标：
  - 延迟（latency）：我们多快能得到单个提示词的答案；
  - 吞吐量（throughput）：在给定时间内我们能处理多少个提示词。
- 单样本生成通常最适合最小化延迟和调试代码
- 批处理主要针对吞吐量
- 如果我们想在 MATH-500 上评估数百个问题、生成许多自一致性样本、或在许多监督样本上训练，批处理可以在合适的硬件上大幅减少总运行时间
  - 但是，批处理并不保证在每个设备上都更快
  - 小模型在 CPU 或某些优化不足的 GPU 上可能不会从批处理中受益；我们甚至可能会变慢，因为额外的填充和批处理开销可能抵消并行化的收益

&nbsp;
## E.2 运行批处理生成

- 批处理中的主要技术障碍是提示词通常有不同的长度
- 例如，一个数学题可能被分词为 40 个 token，而另一个可能被分词为 120 个 token
- 由于 PyTorch 中的张量必须具有矩形形状，我们需要填充较短的序列，使它们都能放入单个批次张量中

- 从概念上讲，这使得批处理生成比单提示词生成更难实现
- 在主要章节中，我们使用了来自 `reasoning_from_scratch.qwen3` 的 `Qwen3Model` 类（它使用附录 C 中解释的 Qwen3 实现）
- 对于批处理生成，由于我们必须跟踪填充 token 等，`reasoning_from_scratch.qwen3_batched` 中有一个单独的 `Qwen3Model` 类（源代码可在补充材料中查看：https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/qwen3_batched.py）

- 为了说明批处理工具的使用，让我们看一个具体的例子
- 我们从一个与主要章节中使用的类似的单序列文本生成示例开始
- 这里，我们将它应用于两个提示词（`["2+2?", "3+3=6?"]`），依次处理：

In [2]:
import torch

from reasoning_from_scratch.ch02 import (
    get_device,
    generate_text_basic_stream_cache,
)
from reasoning_from_scratch.ch03 import (
    load_model_and_tokenizer,
    render_prompt,
)

device = get_device()
model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False,
)

for problem in ["2+2?", "3+3=6?"]:
    prompt = render_prompt(problem)
    input_ids = torch.tensor(
        tokenizer.encode(prompt),
        dtype=torch.long,
        device=device,
    ).unsqueeze(0)

    for token in generate_text_basic_stream_cache(
        model=model,
        token_ids=input_ids,
        max_new_tokens=32,
        eos_token_id=tokenizer.eos_token_id,
    ):
        next_token_id = token.squeeze(0)
        print(tokenizer.decode(next_token_id.tolist()), end="", flush=True)

    print()

Using Apple Silicon GPU (MPS)
✓ qwen3/qwen3-0.6B-base.pth already up-to-date
 \boxed{4}
 \boxed{6}


- 下面，我们将使用来自 `reasoning_from_scratch.qwen3_batched` 的类似代码，它支持批处理
- 注意，批处理版本不支持流式输出，这意味着我们必须等待所有结果生成后才能解码和打印
- 这里，批处理生成使用左填充，这将在下一节中解释
- 现在，让我们从一个使用示例开始，说明如何使用它（在我们深入了解内部工作原理之前）

In [3]:
from reasoning_from_scratch.qwen3_batched import (
    generate_text_basic_batched_cache,
    load_model_and_tokenizer,
)

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False,
)

problems = ["2+2?", "3+3=6?"]
prompts = [render_prompt(problem) for problem in problems]
tokenized = [tokenizer.encode(p) for p in prompts]
pad_id = tokenizer.pad_token_id
max_len = max(len(t) for t in tokenized)

left_padded = [
    [pad_id] * (max_len - len(t)) + t
    for t in tokenized
]
input_ids = torch.tensor(left_padded, dtype=torch.long, device=device)

generated = generate_text_basic_batched_cache(
    model=model,
    token_ids=input_ids,
    max_new_tokens=32,
    eos_token_id=tokenizer.eos_token_id,
    pad_id=pad_id,
)

for row in generated:
    eos_pos = (row == tokenizer.eos_token_id).nonzero(as_tuple=True)[0]
    if len(eos_pos) > 0:
        row = row[:eos_pos[0]]
    print(tokenizer.decode(row.tolist()))

✓ qwen3/qwen3-0.6B-base.pth already up-to-date
 \boxed{4}
 \boxed{6}


- 如我们所见，结果与之前完全相同
- 不同的是，这些结果是通过 `generate_text_basic_batched_cache` 并行生成的
- 下一节简要解释其内部工作原理

- 一个更优化的代码实现是用 `generate_text_basic_batched_cache_stop` 替换 `generate_text_basic_batched_cache`
- `generate_text_basic_batched_cache` 在每个解码步骤中保留活动批次中的每一行
- `generate_text_basic_batched_cache_stop` 从活动计算批次中移除已完成的行（内部实现更复杂，但可以优化性能）
- 这在下图中有所说明

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-e/Appendix_E_F02_raschka.webp?1" width="500px">

- 附注：在 Qwen3 中，`<eos>` token 是 `<|endoftext|>`，但图中为了视觉紧凑使用了 `<eos>`

&nbsp;
## E.3 填充和注意力掩码

- 在单样本模式下，如果我们对一个短提示词（如 `"2+2?"`）进行分词，我们可以将其作为形状为 `(1, 4)` 的简单张量传递给模型：
  - `input_ids = torch.tensor([[17, 10, 17, 30]])`

- 在内部，模型会自动构建一个标准的因果注意力掩码，使得每个位置只能关注自身和之前的 token
- 如果你不熟悉自注意力，我有一篇文章提供了更多背景信息：https://magazine.sebastianraschka.com/p/understanding-and-coding-self-attention
- 从概念上讲，该掩码看起来像这样：

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-e/Appendix_E_F03_raschka.webp" width="400px">

- `1` 表示“被掩码”，`0` 表示“允许”
- 因此第一个 token 不能向前看到后面的位置，第二个 token 只能看到前两个位置，以此类推
- 这是标准的自回归掩码模式

- 批处理改变了这种情况，因为不同的提示词通常有不同的长度
- 假设我们将 `"2+2?"` 与稍长的提示词 `"3+3=6?"` 一起处理
- 由于 PyTorch 张量必须是矩形的，较短的行必须填充以匹配较长的行
- 这里，这是通过左填充完成的：

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-e/Appendix_E_F04_raschka.webp" width="500px">

- 注意，我们在内部保留了一个额外的 `attn_mask`；这只是为了跟踪填充位置
- 在这个 `attn_mask` 中，`True` 表示已填充，`False` 表示未填充
- 我们使用这个额外的 `attn_mask` 来识别因果掩码中对应于填充 token ID 的 token
- 对填充的键进行掩码和将填充的查询置零是使批处理行为与单样本执行类似的重要步骤

- 顺便说一下，我们使用 `<|endoftext|>` token，但这并不重要，因为相应的 token 位置无论如何都会被忽略

In [4]:
print(tokenizer.pad_token_id)

151643


In [5]:
print(tokenizer.decode([151643]))

<|endoftext|>


&nbsp;
## E.4 第 3 章：批处理 MATH-500 评估

- 补充材料包含一个用于第 3 章中实现的评估方法的脚本，我们可以像在第 6 章中那样下载和使用：

In [7]:
from reasoning_from_scratch.ch07 import download_from_github

download_from_github(
    "ch03/02_math500-verifier-scripts/evaluate_math500.py"
)
download_from_github(
    "ch03/01_main-chapter-code/math500_test.json",
    out="math500_test.json",
)

evaluate_math500.py: 3.5 KB
math500_test.json: 462.1 KB


- 然后，要运行它，我们可以在代码终端中执行以下命令（如果你不是 uv 用户，将 `uv run` 替换为 `python`）：

```bash
uv run evaluate_math500.py \
  --dataset_size 500 \
  --which_model "reasoning"
```

- 额外材料还包含一个用于批处理生成的版本，它应用了我们之前讨论的批处理方法
- 下载方式与之前类似，只是我们将 `evaluate_math500.py` 替换为 `evaluate_math500_batched.py`

In [8]:
download_from_github(
    "ch03/02_math500-verifier-scripts/evaluate_math500_batched.py"
)

evaluate_math500_batched.py: 8.3 KB


- 使用方式也与非批处理版本类似，只是我们现在提供一个额外的 `--batch_size` 参数来指定 LLM 应并行处理多少个提示词和答案

```bash
uv run evaluate_math500_batched.py \
  --dataset_size 500 \
  --which_model "reasoning" \
  --batch_size 64
```

- 理想的批处理大小取决于你的硬件能处理什么；批处理大小为 64 时大约使用 23.39 GB RAM（非批处理脚本大约使用 1.84 GB RAM）
- 我们将在附录末尾比较和讨论性能差异

&nbsp;
## E.5 第 4 章：批处理自一致性采样

- 在第 4 章中实现自一致性采样的可选 `self_consistency_math500_batched.py` 脚本不会将不同的提示词混合到一个填充张量中
- 相反，它将相同的提示词重复 `num_samples` 次，并为自一致性投票并行采样多个续写
- 因为每一行都从相同的提示词长度开始，这个脚本使用来自 reasoning_from_scratch.qwen3 的常规 `Qwen3Model`，而不是 reasoning_from_scratch.qwen3_batched，因为相同提示词长度不需要填充

- 我们可以按如下方式下载脚本：

In [ ]:
download_from_github(
    "ch04/02_math500-inference-scaling-scripts/self_consistency_math500_batched.py"
)

- 要下载非批处理版本，只需在上面的文件名中去掉 `"_batched"`
- 我们可以按如下方式运行脚本（非批处理脚本的语法相同）

```bash
uv run self_consistency_math500_batched.py \
  --which_model base \
  --temperature 0.9 \
  --top_p 0.9 \
  --num_samples 3 \
  --dataset_size 500 \
  --prompt_suffix "\n\nExplain step by step."
```

- 有关性能的更多内容请参见本附录末尾

&nbsp;
## E.6 第 6 章：批处理 GRPO 轮次

- 第 5 章中的自我精炼是一种顺序技术，本身不会从批处理中受益
- 可以为多个输入并行运行自我精炼循环，但这实现起来并不简单，因此不包含在补充材料中
- 相反，我们继续介绍第 6 章中 RLVR 的批处理版本
- 在第 6 章中，我们对不同的轮次使用相同的提示词；因此，这里不需要填充；所以，与 E.5 节类似，代码使用来自 `reasoning_from_scratch.qwen3` 的常规 `Qwen3Model` 类
- 相关脚本可以通过以下方式获取：

In [ ]:
# 非批处理版本
download_from_github(
    "ch06/02_rlvr_grpo_scripts_intro/rlvr_grpo_original_no_kl.py"
)

# 批处理版本
download_from_github(
    "ch06/02_rlvr_grpo_scripts_intro/rlvr_grpo_original_no_kl_batched.py"
)

# 批处理版本 with GPU support
download_from_github(
    "ch06/02_rlvr_grpo_scripts_intro/rlvr_grpo_original_no_kl_batched_fsdp.py"
)

```bash
uv run rlvr_grpo_original_no_kl_batched.py \
  --num_rollouts 8 \
  --steps 100 \
  --batch_size 4 \
  --max_new_tokens 1024
```

- 在当前脚本中，`--batch_size` 控制在一个步骤内并行生成多少个轮次
- 这会提高吞吐量，但也会增加内存压力，因此在实践中你可能需要减少 `--num_rollouts` 或 `--max_new_tokens`
- 如果你有多个 GPU，FSDP 变体遵循相同的模式并添加了 `--num_gpus`
- 同样，我们将在附录末尾回到性能讨论
- 截至撰写时，第 7 章脚本的批处理版本尚未在补充材料中提供，但会随时间添加；从概念上讲，它们将与第 6 章脚本类似地工作

&nbsp;
## E.7 第 8 章：批处理蒸馏

- 第 8 章回到了第 3 章的填充感知风格，因为蒸馏样本具有不同的提示词和答案长度
- 你可以按如下方式下载脚本和示例训练数据集：

In [9]:
from reasoning_from_scratch.ch08 import load_distill_data

download_from_github(
    "ch08/04_train_with_distillation/distill_batched.py"
)
_ = load_distill_data(
    partition="deepseek-r1-math-train",
    local_path="deepseek-r1-math-train.json",
)

distill_batched.py: 17.9 KB
deepseek-r1-math-train.json: 107538.0 KB


- 对于非批处理版本，在文件名中去掉 `"_batched"`
- 我们可以按如下方式运行脚本：

```bash
uv run distill_batched.py \
  --data_path deepseek-r1-math-train.json \
  --dataset_size 12000 \
  --validation_size 10 \
  --epochs 2 \
  --use_think_tokens \
  --max_seq_len 1024 \
  --batch_size 4
```

&nbsp;
## E.8 单序列与批处理生成的比较

- 下表总结了上述脚本的运行时间和 RAM 使用数据

| 行 | 脚本                                     | 批处理大小 | RAM      | H100 总时间（分钟） | DGX Spark 总时间（分钟） |
|-----|------------------------------------------|------------|----------|------------------------|-----------------------------|
| 1   | evaluate_math500.py                      | -          | 1.8 GB   | 90.0                   | 174.7                       |
| 2   | evaluate_math500_batched.py              | 64         | 23.39 GB | 16.0                   | 108.4                       |
|     |                                          |            |          |                        |                             |
| 3   | self_consistency_math500.py              | -          | 1.79 GB  | 252.0                  | 340.8                       |
| 4   | self_consistency_math500_batched.py      | 3          | 2.45 GB  | 129.0                  | 243.3                       |
|     |                                          |            |          |                        |                             |
| 5   | rlvr_grpo_original_no_kl.py              | -          | 43.35 GB | 68.0                   | 63.7                        |
| 6   | rlvr_grpo_original_no_kl_batched.py      | 4          | 44.91 GB | 19.0                   | 23.1                        |
|     |                                          |            |          |                        |                             |
| 7   | distill.py                              | -          | 8.29 GB  | 10.9                   | 32.8                        |
| 8   | distill_batched.py                      | 4          | 8.34 GB  | 9.1                    | 28.2                        |